In [1]:
# Prepare dataframe for bias correcting tmax and tmin on a daily scale
# V1 - 2026-07-27 - create code - jpk

In [2]:

# libraries
import os
import gc
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.spatial import cKDTree
from tqdm import tqdm
from rasterio.enums import Resampling

# configurations
start_year, start_month, start_day = 2001, 1, 1
end_year,   end_month, end_day   = 2005, 12, 31

# paths
static_paths = {
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

# In-situ
insitu_tmax_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_maxtemp_1991_2020.csv")
insitu_tmin_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_mintemp_1991_2020.csv")
tmax_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmax.csv"
tmax_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmax.csv"

tmin_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmin.csv"
tmin_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmin.csv"

# Output
out_base = r"..\Data_Temp\Processed\Daily_bias_correction"

tmax_daily_train_dir  = os.path.join(out_base, "tmax_daily_dynamic_train_df")
tmin_daily_train_dir  = os.path.join(out_base, "tmin_daily_dynamic_train_df")

tmax_daily_test_dir  = os.path.join(out_base, "tmax_daily_dynamic_test_df")
tmin_daily_test_dir  = os.path.join(out_base, "tmin_daily_dynamic_test_df")

tmax_parquet_train_dir = os.path.join(out_base, "tmax_yearly_parquet_train")
tmin_parquet_train_dir = os.path.join(out_base, "tmin_yearly_parquet_train")

tmax_parquet_test_dir = os.path.join(out_base, "tmax_yearly_parquet_test")
tmin_parquet_test_dir = os.path.join(out_base, "tmin_yearly_parquet_test")

static_dir = os.path.join(out_base, "static_df")

for d in [tmax_daily_train_dir, tmin_daily_train_dir, tmax_daily_test_dir,
    tmin_daily_test_dir, tmax_parquet_train_dir, tmin_parquet_train_dir,
    tmax_parquet_test_dir, tmin_parquet_test_dir, static_dir]:
    os.makedirs(d, exist_ok=True)

template = rxr.open_rasterio(r"..\Data_Temp\Datasets\CHIRTS\CHIRTS_tmax_20010101.tif", masked = True)
template.rio.write_crs("EPSG:4326", inplace=True)

<xarray.DataArray (band: 1, y: 89, x: 177)> Size: 63kB
[15753 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 712B 30.58 30.53 30.48 30.43 ... 26.28 26.23 26.18
  * x            (x) float64 1kB 79.68 79.73 79.78 79.83 ... 88.38 88.43 88.48
    spatial_ref  int64 8B 0
Attributes: (12/15)
    TIFFTAG_XRESOLUTION:     1
    TIFFTAG_YRESOLUTION:     1
    TIFFTAG_RESOLUTIONUNIT:  1 (unitless)
    AREA_OR_POINT:           Area
    STATISTICS_COUNT:        15753.0
    STATISTICS_COVARIANCES:  46.57341336610955
    ...                      ...
    STATISTICS_MINIMUM:      -0.89422917366028
    STATISTICS_SKIPFACTORX:  1
    STATISTICS_SKIPFACTORY:  1
    STATISTICS_STDDEV:       6.8244716547224
    scale_factor:            1.0
    add_offset:              0.0

In [3]:
# Helpers
def doy_cyclical(year: int, doy: int):
    """Return (sin, cos) encoding for the day of the year."""
    days_in_year = 366 if pd.Timestamp(year, 1, 1).is_leap_year else 365
    angle = 2 * np.pi * doy / days_in_year
    return np.sin(angle), np.cos(angle)

def day_label(year: int, month: int, day: int) -> str:
    return f"{year}{month:02d}{day:02d}"

def build_daily_paths(year: int, month: int, day: int, t_variable: str) -> dict:
    """
    Return dict of {feature_name: filepath} for all daily rasters
    for the given year-month-day. Missing files are noted but not fatal —
    the loader handles absent paths with NaN fill.
    """
    return {
        #Gridded daily temperature products
        f"chirts_{t_variable}": fr"..\Data_Temp\Datasets\CHIRTS\CHIRTS_{t_variable}_{year}{month:02d}{day:02d}.tif",
        f"har_{t_variable}": fr"..\Data_Temp\Datasets\HAR\HAR_{t_variable}_{year}{month:02d}{day:02d}.tif",
        f"era5l_{t_variable}": fr"..\Data_Temp\Datasets\ERA5L\ERA5L_{t_variable}_{year}{month:02d}{day:02d}.tif", 

        #GLDAS weather variables
        "gldas_air_temp": fr"..\Data\Datasets\GLDAS\air_temp\GLDAS_air_temp_{year}{month:02d}{day:02d}.tif",
        "gldas_lwdown": fr"..\Data\Datasets\GLDAS\lwdown\GLDAS_lw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_specific_humidity": fr"..\Data\Datasets\GLDAS\specific_humidity\GLDAS_specific_humidity_{year}{month:02d}{day:02d}.tif",
        "gldas_surface_pressure": fr"..\Data\Datasets\GLDAS\surface_pressure\GLDAS_surface_pressure_{year}{month:02d}{day:02d}.tif",
        "gldas_swdown": fr"..\Data\Datasets\GLDAS\swdown\GLDAS_sw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_wind_speed": fr"..\Data\Datasets\GLDAS\wind_speed\GLDAS_wind_speed_{year}{month:02d}{day:02d}.tif",

        # Insitu kriged
        f"ik_{t_variable}": fr"..\Data_Temp\Datasets\Kriging_daily\{t_variable}\{t_variable}_{year}{month:02d}{day:02d}.tif"
    }    

In [4]:
# 1 Build a static csv
static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template, resampling=Resampling.nearest).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
static_df[["x", "y"]] = static_df[["x", "y"]].round(5)
print(f"Static df: {len(static_df):,} pixels")

Building static df from rasters...
Static df: 15,753 pixels


In [5]:
# 2 Read insitu and separate train-test

# process tmax ==
insitu_tmax_raw["date"]  = pd.to_datetime(insitu_tmax_raw[["year", "month", "day"]])
insitu_tmax_raw  = insitu_tmax_raw.rename(columns={"lat": "y", "long": "x",
    "max_temp": "tmax"})

# Filter for study period
insitu_tmax_daily = insitu_tmax_raw[
    (insitu_tmax_raw["year"]  >= start_year) &
    (insitu_tmax_raw["year"]  <= end_year)
].copy()
insitu_tmax_daily[["x", "y"]] = insitu_tmax_daily[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_daily_train = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_daily_test = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(test_ids_tmax)].copy()

# process tmin ==
insitu_tmin_raw["date"]  = pd.to_datetime(insitu_tmin_raw[["year", "month", "day"]])
insitu_tmin_raw  = insitu_tmin_raw.rename(columns={"lat": "y", "long": "x",
    "min_temp": "tmin"})

# Filter for study period
insitu_tmin_daily = insitu_tmin_raw[
    (insitu_tmin_raw["year"]  >= start_year) &
    (insitu_tmin_raw["year"]  <= end_year)
].copy()
insitu_tmin_daily[["x", "y"]] = insitu_tmin_daily[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_daily_train = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_daily_test = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(test_ids_tmin)].copy()

In [ ]:
# 3 Daily loop to aggregate daily dataframes

daily_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}-{start_day:02d}",
    end = f"{end_year}-{end_month:02d}-{end_day:02d}",
    freq= "D"
)

print(f"\n ============ Processing {len(daily_range)} days for Tmax =========")

for period in tqdm(daily_range, desc = "Daily feature extraction"):
    year = period.year
    month = period.month
    day = period.day
    label = day_label(year, month, day)

    tmax_train_out = os.path.join(tmax_daily_train_dir, f"tmax_daily_{label}.csv")
    tmax_test_out = os.path.join(tmax_daily_test_dir, f"tmax_daily_{label}.csv")

    if os.path.exists(tmax_train_out) and os.path.exists(tmax_test_out):
        continue

    # A Load daily rasters and flatten to pixel df
    paths = build_daily_paths(year, month, day, "tmax")
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked = True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template, resampling = Resampling.nearest).squeeze("band", drop = True)
        da.name =  name
        aligned.append(da.load())
        da.close()
    
    if missing_vars:
        tqdm.write(f"   [{label}] missing rasters:  {missing_vars}")

    if not aligned:
        tqdm.write(f"   [{label}]  no rasters found - skipping")
        continue
    
    # B Merge all dynamic rasters to a dataframe
    dynamic_xr = xr.merge(aligned, compat = 'override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[['x', 'y']] = dynamic_df[['x', 'y']].round(5)

    for name in missing_vars:
        dynamic_df[name] = np.nan
    
    del aligned, dynamic_xr
    gc.collect()

    # C Merge with static feature
    daily_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = 'inner')
    doy = pd.to_datetime({'year': [year], 'month': [month], 'day': [day]})[0].dayofyear
    sin_doy, cos_doy = doy_cyclical(year, doy)

    daily_grid['doy_sin'] = sin_doy
    daily_grid['doy_cos'] = cos_doy
    daily_grid['year'] = year
    daily_grid['month'] = month
    daily_grid['day'] = day

    del dynamic_df
    gc.collect()

    # D cKDTree match to station locations
    # Threshold 0.006° ≈ ~600m at Nepal latitudes — one pixel tolerance.
    insitu_train_day = insitu_tmax_daily_train[
        (insitu_tmax_daily_train['year'] == year) &
        (insitu_tmax_daily_train['month'] == month) &
        (insitu_tmax_daily_train['day'] == day)
    ].copy()

    insitu_test_day = insitu_tmax_daily_test[
        (insitu_tmax_daily_test['year'] == year) &
        (insitu_tmax_daily_test['month'] == month) &
        (insitu_tmax_daily_test['day'] == day) 
    ].copy()

    if insitu_train_day.empty and insitu_test_day.empty:
        del daily_grid
        gc.collect()
        continue
    
    tree = cKDTree(daily_grid[['x', 'y']].values)

    if not insitu_train_day.empty:
        dist_tr, idx_tr = tree.query(insitu_train_day[['x', 'y']].values, k=1)
        mask_tr = dist_tr <= 0.035

        if mask_tr.any():
            station_rows = insitu_train_day.iloc[mask_tr].reset_index(drop=True)
            grid_rows = daily_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)

            final_train = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "tmax"]],
                grid_rows.drop(columns= ['x', 'y', 'year', 'month', 'day'])
            ], axis=1)

            final_train.to_csv(tmax_train_out, index = False)
    
    if not insitu_test_day.empty:
        dist_te, idx_te = tree.query(insitu_test_day[['x', 'y']].values, k=1)
        mask_te = dist_te <= 0.035

        if mask_te.any():
            station_rows = insitu_test_day.iloc[mask_te].reset_index(drop=True)
            grid_rows = daily_grid.iloc[idx_te[mask_te]].reset_index(drop=True)

            final_test = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "tmax"]],
                grid_rows.drop(columns=  ['x', 'y', 'year', 'month', 'day'])
            ], axis = 1)
            final_test.to_csv(tmax_test_out, index = False)
    
    del daily_grid, insitu_train_day, insitu_test_day
    gc.collect()


 ============ Processing 1826 days for Tmax =========


Daily feature extraction:   0%|          | 0/1826 [00:00<?, ?it/s]

Daily feature extraction:   7%|▋         | 132/1826 [01:10<14:47,  1.91it/s]

In [ ]:
print(f"\n ============ Processing {len(daily_range)} days for Tmin =========")

for period in tqdm(daily_range, desc = "Daily feature extraction"):
    year = period.year
    month = period.month
    day = period.day
    label = day_label(year, month, day)

    tmin_train_out = os.path.join(tmin_daily_train_dir, f"tmin_daily_{label}.csv")
    tmin_test_out = os.path.join(tmin_daily_test_dir, f"tmin_daily_{label}.csv")

    if os.path.exists(tmin_train_out) and os.path.exists(tmin_test_out):
        continue

    # A Load daily rasters and flatten to pixel df
    paths = build_daily_paths(year, month, day, "tmin")
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked = True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop = True)
        da.name =  name
        aligned.append(da.load())
        da.close()
    
    if missing_vars:
        tqdm.write(f"   [{label}] missing rasters:  {missing_vars}")

    if not aligned:
        tqdm.write(f"   [{label}]  no rasters found - skipping")
        continue
    
    # B Merge all dynamic rasters to a dataframe
    dynamic_xr = xr.merge(aligned, compat = 'override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[['x', 'y']] = dynamic_df[['x', 'y']].round(5)

    for name in missing_vars:
        dynamic_df[name] = np.nan
    
    del aligned, dynamic_xr
    gc.collect()

    # C Merge with static feature
    daily_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = 'inner')
    doy = pd.to_datetime({'year': [year], 'month': [month], 'day': [day]})[0].dayofyear
    sin_doy, cos_doy = doy_cyclical(year, doy)

    daily_grid['doy_sin'] = sin_doy
    daily_grid['doy_cos'] = cos_doy
    daily_grid['year'] = year
    daily_grid['month'] = month
    daily_grid['day'] = day

    del dynamic_df
    gc.collect()

    # D cKDTree match to station locations
    # Threshold 0.035° ≈  at Nepal latitudes — one-half diagonal distance of grid resolution (1 pixel tolerance).
    insitu_train_day = insitu_tmin_daily_train[
        (insitu_tmin_daily_train['year'] == year) &
        (insitu_tmin_daily_train['month'] == month) &
        (insitu_tmin_daily_train['day'] == day) 
    ].copy()

    insitu_test_day = insitu_tmin_daily_test[
        (insitu_tmin_daily_test['year'] == year) &
        (insitu_tmin_daily_test['month'] == month) &
        (insitu_tmin_daily_test['day'] == day)
    ].copy()

    if insitu_train_day.empty and insitu_test_day.empty:
        del daily_grid
        gc.collect()
        continue
    
    tree = cKDTree(daily_grid[['x', 'y']].values)

    if not insitu_train_day.empty:
        dist_tr, idx_tr = tree.query(insitu_train_day[['x', 'y']].values, k=1)
        mask_tr = dist_tr <= 0.035

        if mask_tr.any():
            station_rows = insitu_train_day.iloc[mask_tr].reset_index(drop=True)
            grid_rows = daily_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)

            final_train = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "tmin"]],
                grid_rows.drop(columns= ['x', 'y', 'year', 'month', 'day'])
            ], axis=1)

            final_train.to_csv(tmin_train_out, index = False)
    
    if not insitu_test_day.empty:
        dist_te, idx_te = tree.query(insitu_test_day[['x', 'y']].values, k=1)
        mask_te = dist_te <= 0.035

        if mask_te.any():
            station_rows = insitu_test_day.iloc[mask_te].reset_index(drop=True)
            grid_rows = daily_grid.iloc[idx_te[mask_te]].reset_index(drop=True)

            final_test = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month", "day", "tmin"]],
                grid_rows.drop(columns=  ['x', 'y', 'year', 'month', 'day'])
            ], axis = 1)
            final_test.to_csv(tmin_test_out, index = False)
    
    del daily_grid, insitu_train_day, insitu_test_day
    gc.collect()


 ============ Processing 1796 days for Tmin =========


Daily feature extraction: 100%|██████████| 1796/1796 [12:06<00:00,  2.47it/s]


In [ ]:
# 5 Aggregate monthly CSVs to yearly parquet --> final combined df

def aggregate_to_parquet(daily_dir: str, parquet_dir: str, t_variable: str, split_label:str) -> str:
    """
    Read daily csvs year by year, write to parquet, then concatenate
    all years into a single final df. Returns path to final parquet
    """
    parquet_paths = []

    for year in range(start_year, end_year + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        daily_periods = pd.period_range(
            start = f"{year}-01-01", end = f"{year}-12-31", freq = "D"
        )
        expected = [
            os.path.join(daily_dir, f"{t_variable}_daily_{day_label(p.year, p.month, p.day)}.csv")
            for p in daily_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"  {t_variable} - [{split_label}]  {year} - no daily files found, skipping")
            continue

        writer = None
        for fpath in found:
            chunk = pd.read_csv(fpath)
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(yearly_path, table.schema)
            writer.write_table(table)

        if writer:
            writer.close()
            parquet_paths.append(yearly_path)
            print(f" [{split_label}] {year} - {len(found)} days written to parquet")
    
    if not parquet_paths:
        print(f"    [{split_label}] No data - final df not written")
        return None
    
    final_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index=True)
    final_path = os.path.join(out_base, f"final_{t_variable}_{split_label}_df.parquet")
    final_df.to_parquet(final_path, index = False)

    print(f"\n  [{t_variable} - {split_label}] Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path

tmax_train_final_path = aggregate_to_parquet(tmax_daily_train_dir, tmax_parquet_train_dir, "tmax", "train")
tmax_test_final_path = aggregate_to_parquet(tmax_daily_test_dir, tmax_parquet_test_dir, "tmax", "test")
tmin_train_final_path = aggregate_to_parquet(tmin_daily_train_dir, tmin_parquet_train_dir, "tmin", "train")
tmin_test_final_path = aggregate_to_parquet(tmin_daily_test_dir, tmin_parquet_test_dir, "tmin", "test")

 [train] 2001 - 365 days written to parquet
 [train] 2002 - 365 days written to parquet
 [train] 2003 - 365 days written to parquet
 [train] 2004 - 366 days written to parquet
 [train] 2005 - 335 days written to parquet

  [tmax - train] Final df: 64,449 rows × 33 cols
  Saved: ..\Data_Temp\Processed\Daily_bias_correction\final_tmax_train_df.parquet
 [test] 2001 - 365 days written to parquet
 [test] 2002 - 365 days written to parquet
 [test] 2003 - 365 days written to parquet
 [test] 2004 - 366 days written to parquet
 [test] 2005 - 335 days written to parquet

  [tmax - test] Final df: 40,388 rows × 33 cols
  Saved: ..\Data_Temp\Processed\Daily_bias_correction\final_tmax_test_df.parquet
 [train] 2001 - 365 days written to parquet
 [train] 2002 - 365 days written to parquet
 [train] 2003 - 365 days written to parquet
 [train] 2004 - 366 days written to parquet
 [train] 2005 - 335 days written to parquet

  [tmin - train] Final df: 64,199 rows × 33 cols
  Saved: ..\Data_Temp\Processed\D

In [ ]:
# ── 7. Clean up intermediate files ──────────────────────────────────────────────

import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [
    tmax_daily_train_dir,
    tmin_daily_train_dir,
    tmax_daily_test_dir,
    tmin_daily_test_dir,
    tmax_parquet_train_dir,
    tmin_parquet_train_dir,
    tmax_parquet_test_dir,
    tmin_parquet_test_dir,
    static_dir
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")

# # Also remove preview csvs
# for split in ['train', 'test']:
#     preview_csv = os.path.join(OUT_BASE, f"final_{split}_df_preview.csv")
#     if os.path.exists(preview_csv):
#         os.remove(preview_csv)
#         print(f"Removed preview CSV: {preview_csv}")

print("\nCleanup complete.")



Cleaning up intermediate files and directories...
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmax_daily_dynamic_train_df
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmin_daily_dynamic_train_df
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmax_daily_dynamic_test_df
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmin_daily_dynamic_test_df
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmax_yearly_parquet_train
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmin_yearly_parquet_train
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmax_yearly_parquet_test
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\tmin_yearly_parquet_test
Removed directory: ..\Data_Temp\Processed\Daily_bias_correction\static_df

Cleanup complete.
